# 05. Fixed and aligned windows: current and combined configurations

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.transfer import transfer
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
cohorts = list(transfer.FULL_COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
print("Development features:", features_path)

In [ ]:
augmentation_run = ""  # Optional pipeline_outputs/transfer/augmentation_<id>; blank uses the latest
augmentation_run = Path(augmentation_run) if augmentation_run else tuning.latest_run(root, "augmentation", area=transfer.AREA)
choice = json.loads((augmentation_run / "selected.json").read_text())
print("augmentation run:", augmentation_run)
choice

## Configurations

In [ ]:
configurations = {}
for cohort in cohorts:
    configurations[cohort, "current"] = {"params": tuning.BASELINE, "variant": "interval_72", "augmentation": None}
    configurations[cohort, "combined"] = {"params": choice[cohort]["params"], "variant": choice[cohort]["series_variant"],
                                          "augmentation": choice[cohort]["augmentation_settings"]}
pd.DataFrame([{"cohort": c, "configuration": n, **{k: str(v) for k, v in s.items()}} for (c, n), s in configurations.items()])

## Evaluate with 1,000 trees

In [ ]:
run = tuning.new_run(root, "summary", features_path, area=transfer.AREA, augmentation_run=str(augmentation_run))
records, holdout = [], []
for (cohort, name), setting in configurations.items():
    params = {**setting["params"], "n_estimators": 1000}
    train, validation = tuning.cohort_split(development, cohort)
    pooled = transfer.spatial_cv(train, params, setting["variant"], setting["augmentation"])
    locked = transfer.locked_validation(train, validation, params, setting["variant"], setting["augmentation"])
    scored = [r for r in transfer.leave_one_source_out(train + validation, params, setting["variant"], setting["augmentation"])
              if r["status"] == "scored"]
    holdout.extend({"cohort": cohort, "configuration": name, **r} for r in scored)
    records.append({"cohort": cohort, "configuration": name, "variant": setting["variant"],
                    "augmentation": str(setting["augmentation"]),
                    "spatial_macro_f1": pooled["macro_f1"], "fold_min": min(pooled["fold_macro_f1"]),
                    "fold_max": max(pooled["fold_macro_f1"]), "locked_validation_macro_f1": locked["macro_f1"],
                    "source_holdout_macro_f1_mean": sum(r["macro_f1_scored_crops"] for r in scored) / len(scored) if scored else None,
                    **{f"spatial_{crop}_f1": v["f1"] for crop, v in pooled["per_crop"].items()},
                    **{f"locked_{crop}_f1": v["f1"] for crop, v in locked["per_crop"].items()}})
    print(cohort, name, "done", flush=True)
tuning.save_records(run, "window_summary", records)
tuning.save_records(run, "window_leave_one_source_out", holdout)

## Summary table

In [ ]:
summary = pd.DataFrame(records)
display(summary.round(3))
display(pd.DataFrame(holdout).round(3))

## Figure

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for axis, metric in zip(axes, ["spatial_macro_f1", "locked_validation_macro_f1", "source_holdout_macro_f1_mean"]):
    summary.pivot(index="cohort", columns="configuration", values=metric).plot(kind="bar", ax=axis)
    axis.set_title(metric)
    axis.set_ylim(0, 1)
show_and_save(figure, run, "window_summary")
print("Run folder:", run)